# Proof-of-Concept 4-Parameter Model

This notebook demonstrates a small proof-of-concept emulator that predicts the **binned kSZ angular power spectrum ($D_\ell$)** using 4 reionization params ($z_{mean}$, $\alpha$, $k_b$, $b_0$).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

plt.rc("figure", figsize=(6, 4), dpi=150)

from reionemu import (
    DataLoaderConfig,
    FitConfig,
    FourParamEmulator,
    fit,
    load_training_arrays,
    make_dataloaders,
)

## File Paths
The condensed v6 simulation dataset is constructed with: $Y = \ln(D_\ell)$

In [ ]:
H5_PATH = Path("../data/processed/condensed_v6.h5").resolve()
MODEL_PATH = Path("../checkpoints/poc_four_params/model.pt").resolve()
NORM_PATH = Path("../checkpoints/poc_four_params/norm/").resolve()

## Define Configs

In [ ]:
dlcfg = DataLoaderConfig(
    batch_size=32, seed=42, shuffle_train=True, normalize_X=True, normalize_Y=False
)

fitcfg = FitConfig(
    epochs=200, device="mps", early_stopping_patience=50, gradient_clipping=None
)

## Prepare Data and Model

In [ ]:
loaders, norms, ell = make_dataloaders(
    H5_PATH, split={"train": 0.8, "val": 0.2}, config=dlcfg
)

model = FourParamEmulator()
lossfn = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

model.eval()

## Train Model

In [ ]:
history = fit(model, loaders["train"], loaders["val"], optimizer, lossfn, config=fitcfg)

## Save Model and Normalization

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

NORM_PATH.parent.mkdir(parents=True, exist_ok=True)
np.save(NORM_PATH / "X_mean.npy", norms["X"].mean)
np.save(NORM_PATH / "X_std.npy", norms["X"].std)
if dlcfg.normalize_Y == True:
    np.save(NORM_PATH / "Y_mean.npy", norms["Y"].mean)
    np.save(NORM_PATH / "Y_std.npy", norms["Y"].std)
np.save(NORM_PATH / "ell", np.asarray(ell))

# Load Model and Normalization

In [ ]:
device = torch.device("mps")

X_mean = np.load(f"{NORM_PATH}/X_mean.npy")
X_std = np.load(f"{NORM_PATH}/X_std.npy")
Y_mean = np.load(f"{NORM_PATH}/Y_mean.npy")
Y_std = np.load(f"{NORM_PATH}/Y_std.npy")
ell = np.load(f"{NORM_PATH}/ell.npy")

model = FourParamEmulator().to(device)
state_dict = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state_dict)

## Define Predict Function

In [ ]:
def predict(params, model, X_mean, X_std, Y_mean=None, Y_std=None, normalize_Y=True):
    params = (params - X_mean) / X_std

    xb = torch.from_numpy(params).to(device)

    model.eval()
    with torch.no_grad():
        pred_norm = model(xb).cpu().numpy()

    if normalize_Y:
        pred_log = pred_norm * Y_std + Y_mean
    else:
        pred_log = pred_norm

    pred_dl = np.exp(pred_log)
    return pred_dl

## Error Metrics
- **Mean Squared Error (MSE)**: Computed in physical units $\mu K^2$ large errors are penalized.
- **Mean Absolute Error (MAE)**: Computed in physical units $\mu K^2$ errors are treated equally.

In [ ]:
X, Y, ell = load_training_arrays(H5_PATH)


pred = []
true = []

if dlcfg.normalize_Y == True:
    pred = predict(X, model, X_mean, X_std, Y_mean, Y_std)
else:
    pred = predict(X, model, X_mean, X_std, normalize_Y=False)


true = np.exp(Y)

errors = np.abs((pred - true) / true)

mse = np.mean((pred - true) ** 2)
mae = np.mean(np.abs((pred - true)))
percent_err = np.mean(errors) * 100

bin_percent_err = np.mean(errors, axis=0) * 100

flat_idx = np.argmax(errors)
i, b = np.unravel_index(flat_idx, errors.shape)

print("Mean Squared Error (MSE):\t", mse)
print("Mean Absolute Error (MAE):\t", mae)
print("Mean % Error:\t\t\t\t", percent_err)
print()
print("Max % Error:\t\t\t\t", np.max(errors) * 100)
print("Mean % Error Per ell Bin:\t", bin_percent_err)
print()
print("Worst Case Sim:\t", i)
print("Worst Case Bin:\t", b)
print("True Dl:\t\t", true[i, b])
print("Pred Dl:\t\t", pred[i, b])
print("% error:\t\t", errors[i, b] * 100)

## Isolate Sim

In [ ]:
sim_idx = 210

X, Y, ell = load_training_arrays(H5_PATH)
params = X[sim_idx]
true_dl = np.exp(Y[sim_idx])
pred_dl = predict(params, model, X_mean, X_std, Y_mean, Y_std)

print(f"Params:\t\t {params}")
print()
print(f"True:\t\t {true_dl}")
print(f"Predicted:\t {pred_dl}")

## True vs Predicted $D_\ell$

In [ ]:
def plot_sim(sim_idx):
    X, Y, ell = load_training_arrays(H5_PATH)

    params = X[sim_idx]
    true_dl = np.exp(Y[sim_idx])
    pred_dl = predict(params, model, X_mean, X_std, Y_mean, Y_std)

    pct_err = 100.0 * (pred_dl - true_dl) / true_dl

    print(f"Sim Index:\t\t{sim_idx}")
    print(f"Params:\t\t\t{params}")
    print(f"True Dl:\t\t{true_dl}")
    print(f"Predicted Dl:\t{pred_dl}")
    print(f"% Error:\t\t{pct_err}")

    plt.plot(ell, true_dl, marker="o", label=r"True $D_\ell$")
    plt.plot(ell, pred_dl, marker="s", label=r"Predicted $D_\ell$")
    plt.xlabel(r"$\ell$ bin center")
    # plt.ylabel(r"$D_\ell$")
    plt.ylabel(r"$\ell (\ell + 1)C_{\ell}$ / $2 \pi$ [$uK^2$]")
    plt.title(f"True vs Predicted $D_\\ell$ (sim {sim_idx})")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_sim(0)
plot_sim(210)